In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, homogeneity_completeness_v_measure
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [2]:
df_train = pd.read_csv("train_sentiment.csv")
df_test = pd.read_csv("test_sentiment.csv")

In [3]:
train_X, train_y, test_X, test_y = df_train["review"], df_train["sentiment"], df_test["review"], df_test["sentiment"]
train_X, val_X, train_y, val_y = train_test_split(train_X, train_y, test_size=0.2)

In [4]:
def build_vocab(texts, max_vocab_size):
    counter = Counter()

    for sentence in texts:
        counter.update(sentence.split())

    # Special tokens
    vocab = {
        "<PAD>": 0,
        "<OOV>": 1
    }

    for idx, (word, _) in enumerate(
        counter.most_common(max_vocab_size - 2), start=2
    ):
        vocab[word] = idx

    return vocab

In [5]:
max_vocab_size = 10000
vocab = build_vocab(train_X.values, max_vocab_size)

In [6]:
def texts_to_sequences(texts, vocab):
    sequences = []

    for sentence in texts:
        seq = [
            vocab.get(word, vocab["<OOV>"])
            for word in sentence.split()
        ]
        sequences.append(seq)

    return sequences

In [7]:
train_X = texts_to_sequences(train_X.values, vocab)
val_X   = texts_to_sequences(val_X.values, vocab)
test_X  = texts_to_sequences(test_X.values, vocab)

In [8]:
def pad(sequences, pad_value=0):
    return pad_sequence(
        [torch.tensor(seq, dtype=torch.long) for seq in sequences],
        batch_first=True,
        padding_value=pad_value
    )

In [9]:
train_X = pad(train_X, pad_value=0)
val_X   = pad(val_X,pad_value=0)
test_X  = pad(test_X, pad_value=0)
train_y = torch.tensor(train_y.values, dtype=torch.float32)
val_y   = torch.tensor(val_y.values, dtype=torch.float32)
test_y  = torch.tensor(test_y.values, dtype=torch.float32)

In [10]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.conv1 = nn.Conv1d(embedding_dim, 128, kernel_size=3, padding=2)
        self.conv2 = nn.Conv1d(128, 64, kernel_size=3, padding=2)

        self.fc1 = nn.Linear(64, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embedding(x)          
        x = x.permute(0, 2, 1)        

        x = self.conv1(x)
        x = self.conv2(x)
        x = torch.max(x, dim=2).values
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x).squeeze(1)

In [11]:
model = TextCNN(vocab_size=len(vocab), embedding_dim=64)

In [12]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)